# Track-aware multi-emitter observation series + intelligence-report generation (v2)

This notebook generates synthetic single-aircraft emitter tracks and enriches each series with shared **sighting reports** and **patterns-of-life reports**. Sightings describe nearby aircraft, operators, radars, geographic positions, and last-observed times, with a mostly-correct mix that includes controlled errors. Patterns of life describe expected aircraft-family/variant, operator, radar, role, and radar-mode behaviours. Ground-truth fields remain available only for evaluation.

The combined generator creates each ESM series and its intelligence reports together in the same worker process. This parallelises report creation, avoids a second parent-process enrichment pass, and transfers each large series between processes only once. Reports and their observation-ID applicability list are stored once per series rather than once per observation. The validation cells below also stream over reports and observations instead of materialising large flattened helper lists.

The series generator also samples the enriched ESM signature fields from each canonical radar mode: PRI modulation, intrapulse modulation, frequency pattern/agility, scan period, and polarization. Numeric values retain measurement uncertainty rather than exposing the canonical mode bounds directly.

Version 2 models four operational emission states (`radar_silent`, `low_emission`, `totally_passive`, and `active`). Radar OFF observations intentionally have no radar ESM, low-emission intercepts can omit individual RF fields, and data-link, radio, and radar-altimeter emissions are independently detectable according to state. Kinematics can be complete, partial/bearing-only, or absent. The adjustable probabilities are explicit in the generation cell. Each report set also includes an airborne-aircraft report selected from the theatre aircraft list.


In [ ]:
from pathlib import Path
import os
from collections import Counter
import gc
from itertools import chain

from esm_observation_series_generator import (
    generate_observation_series_with_intelligence_reports,
    observations_without_ground_truth,
    write_observation_series_json,
)
from rgcn_fusion.intelligence_reports import build_report_evidence_rows


def clean():
    while gc.collect()>0:
        gc.collect()

    return

In [ ]:
# Use multiple worker processes so this demo exercises the parallelized series generator.
# Cap the notebook demo to a few workers to avoid oversubscribing shared environments.
worker_count = min(18, os.cpu_count() or 1)
# Worker processes generate and enrich independent series in one pass, so report
# generation is parallel and each complete series crosses the process boundary once.
# Cap the demo to a few workers to avoid oversubscribing shared environments.
# worker_count = min(14, os.cpu_count() or 1)

data = generate_observation_series_with_intelligence_reports(
    count=16000,
    seed=42,
    intelligence_seed=4200,
    min_duration_s=1.0,
    max_duration_s=45.0,
    sample_interval_s=0.5,
    min_reports_per_series=4,
    max_reports_per_series=15,
    mode_switch_probability=0.08,  # active radar modes may still transition
    radar_off_probability=0.60,  # easily adjusted: high probability requested for radar OFF
    kinematic_dropout_probability=0.25,  # partial data, frequently bearing-only
    all_kinematic_dropout_probability=0.10,  # no kinematic fields at all
    workers=worker_count,
)

# Stream summaries rather than allocating flattened report and claim lists.
reports = chain.from_iterable(
    series["intelligence_reports"] for series in data["observation_series"]
)
report_count = 0
report_types = Counter()
claim_types = Counter()
for report in reports:
    report_count += 1
    report_types[report["report_type"]] += 1
    claim_types.update(claim["claim_type"] for claim in report["claims"])

# Every series gets one shared report set; observations retain measurements only.
for series in data["observation_series"]:
    series_reports = series["intelligence_reports"]
    observation_ids = [obs["observation_id"] for obs in series["observations"]]
    assert all("esm_radar_parameters" in obs and "approximate_kinematics" in obs for obs in series["observations"])
    assert all(obs["operational_emission_state"] in {"radar_silent", "low_emission", "totally_passive", "active"} for obs in series["observations"])
    assert all("intelligence_reports" not in obs for obs in series["observations"])
    assert all(report["valid_for_observation_ids"] == observation_ids for report in series_reports)
len(data["observation_series"]), data["metadata"], report_count, report_types, claim_types


In [ ]:
# Verify enriched parameters without retaining a flattened list of every ESM row.
enriched_fields = (
    "observed_pri_modulation",
    "observed_intrapulse_modulation",
    "observed_frequency_pattern",
    "observed_polarization",
    "measured_frequency_agility_mhz",
    "measured_scan_period_s",
)
esm_rows = (
    observation["esm_radar_parameters"]
    for series in data["observation_series"]
    for observation in series["observations"]
    if observation["esm_radar_parameters"] is not None
)
first_esm_row = next(esm_rows)
assert all(field in first_esm_row for field in enriched_fields)
# Low-emission rows are intentionally incomplete, while active rows remain complete.
active_rows = [observation["esm_radar_parameters"] for series in data["observation_series"] for observation in series["observations"] if observation["operational_emission_state"] == "active"]
assert all(all(field in esm for field in enriched_fields) for esm in active_rows)
{field: first_esm_row[field] for field in enriched_fields}


In [ ]:
data["observation_series"][0]

In [ ]:
first_series = data["observation_series"][0]
inference_rows = observations_without_ground_truth(first_series)
first_obs = first_series["observations"][0]
first_reports = first_series["intelligence_reports"]
{
    "series_id": first_series["series_id"],
    "observation_count": first_series["observation_count"],
    "shared_report_count": len(first_reports),
    "report_types": Counter(report["report_type"] for report in first_reports),
    "first_sighting": next(report["sighting"] for report in first_reports if report["report_type"] == "sighting_report"),
    "known_airborne": next(report["airborne_aircraft"] for report in first_reports if report["report_type"] == "airborne_aircraft_report"),
    "first_pattern_of_life": next(report["pattern_of_life"] for report in first_reports if report["report_type"] == "pattern_of_life_report"),
    "report_valid_for_every_observation": set(first_reports[0]["valid_for_observation_ids"]) == {obs["observation_id"] for obs in first_series["observations"]},
    "radar_off_observation_count": sum(obs["esm_radar_parameters"] is None for obs in first_series["observations"]),
    "bearing_only_or_partial_count": sum(set(obs["approximate_kinematics"]) != {"ground_speed_kph", "ground_speed_error_kph", "ground_speed_min_kph", "ground_speed_max_kph", "altitude_m", "altitude_error_m", "altitude_min_m", "altitude_max_m", "heading_deg", "heading_error_deg"} for obs in first_series["observations"]),
    "observations_do_not_duplicate_reports": all("intelligence_reports" not in obs for obs in first_series["observations"]),
    "inference_row_has_ground_truth": "ground_truth_label" in inference_rows[0],
}


In [ ]:
evidence_rows = build_report_evidence_rows(data["observation_series"])
{name: len(rows) for name, rows in evidence_rows.items()}


In [ ]:
output_path = Path("../generated/demo_esm_observation_series_multimode_v2.json")
write_observation_series_json(data, output_path)
output_path
